In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 1. Загрузим данные
df = pd.read_csv('mydata.csv')  # предполагаем, что здесь labels — 0 и 1

# 2. Преобразуем метки 0→"human", 1→"AI"
label_mapping = {0: 'human', 1: 'AI'}
df['label'] = df['label'].map(label_mapping)

# 3. Разбиваем на train и test
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'],
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

In [ ]:
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50_000, ngram_range=(1,2))),
    ('nb', MultinomialNB(alpha=0.7))
])

In [ ]:
nb_pipeline.fit(X_train, y_train)

In [ ]:
y_pred = nb_pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(
    classification_report(
        y_test,
        y_pred,
        labels=['human', 'AI'],
        target_names=['human', 'AI'],
        digits=4
    )
)

In [ ]:
# 8. Строим матрицу ошибок
#    Здесь берем порядок меток ['human', 'AI'] для осей
cm_labels = ['human', 'AI']
cm = confusion_matrix(y_test, y_pred, labels=cm_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=cm_labels,   # подписи по оси X: предсказанные
    yticklabels=cm_labels    # подписи по оси Y: истинные
)
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.title('Матрица ошибок НБК')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_curve, auc
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Загрузим данные и заменим метки 0→"human", 1→"AI"
df = pd.read_csv('mydata.csv')
label_mapping = {0: 'human', 1: 'AI'}
df['label'] = df['label'].map(label_mapping)

# 2. Разбиваем на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'],
    df['label'],
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

# 3. Строим пайплайн TF-IDF + MultinomialNB (NB уже обучен ранее)
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50_000, ngram_range=(1,2))),
    ('nb', MultinomialNB(alpha=0.7))
])
nb_pipeline.fit(X_train, y_train)

# 4. Проверка, что классов ровно два
if len(nb_pipeline.classes_) != 2:
    raise ValueError(
        "ROC‐кривую можно рисовать только для бинарной классификации. "
        f"У вас {len(nb_pipeline.classes_)} классов."
    )

# 5. Предсказанные вероятности для «положительного» класса (classes_[1] → "AI")
y_prob = nb_pipeline.predict_proba(X_test)  # shape = (n_samples, 2)
# Берём столбец с индексом 1, соответствующий «AI»
y_prob_pos = y_prob[:, 1]

# 6. Перекодируем y_test в 0/1, где 1 = «AI», 0 = «human»
mapping = {nb_pipeline.classes_[0]: 0, nb_pipeline.classes_[1]: 1}
y_test_bin = y_test.map(mapping)

# 7. Считаем FPR, TPR, thresholds и AUC
fpr, tpr, thresholds = roc_curve(y_test_bin, y_prob_pos)
roc_auc = auc(fpr, tpr)

# 8. Рисуем ROC‐кривую
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8)  # диагональная линия
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC‐кривая НБК')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# 9. Трансформация всех текстов в TF-IDF
tfidf = nb_pipeline.named_steps['tfidf']
X_all_tfidf = tfidf.transform(df['clean_text'])

# 10. Снижаем размерность через TruncatedSVD (например, до 50 компонент)
svd = TruncatedSVD(n_components=50, random_state=42)
X_svd = svd.fit_transform(X_all_tfidf)

# 11. Применяем t-SNE к результату SVD, чтобы получить 2D-координаты
tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')
X_tsne = tsne.fit_transform(X_svd)

# 12. Строим scatter plot, раскрашивая точки по меткам «human» и «AI»
plt.figure(figsize=(8, 6))
for label in ['human', 'AI']:
    idx = (df['label'] == label)
    plt.scatter(
        X_tsne[idx, 0],
        X_tsne[idx, 1],
        label=label,
        alpha=0.6,
        s=20
    )

plt.xlabel('t-SNE компонентa 1')
plt.ylabel('t-SNE компонентa 2')
plt.title('t-SNE проекция TF-IDF признаков (метки “human” vs “AI”)')
plt.legend(markerscale=2, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()